# 01 — Data Collection & First Similarity Engine

This notebook builds the first working version of the **Champions League-Level Player Discovery** project.

The objective is to identify **non-Champions League strikers** with statistical profiles similar to strikers from Champions League clubs.

Current scope:
- Top 5 European leagues, 2025/2026 season
- Strikers only (`FW`)
- Role-specific per90 feature engineering
- Nearest Neighbors similarity model
- Example scouting report based on **Julián Álvarez**

In [2]:
# Cell 1 — Import required libraries
# pandas and numpy are used for data handling.
# scikit-learn is used for feature scaling and similarity modeling.

import os
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

In [3]:
# Cell 2 — Load raw player dataset
# The dataset is stored locally in the data/raw folder.
# It contains player statistics from Europe's top 5 leagues for the 2025/2026 season.

DATA_PATH = "../data/raw/players_data_light-2025_2026.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,Save%,W,D,L,CS,CS%,PKatt_stats_keeper,PKA,PKsv,PKm
0,1,Brenden Aaronson,us USA,"MF,FW",Leeds United,eng Premier League,25.0,2000.0,34,27,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Zach Abbott,eng ENG,DF,Nottingham Forest,eng Premier League,19.0,2006.0,3,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Jones El-Abdellaoui,ma MAR,"MF,FW",Celta Vigo,es La Liga,20.0,2006.0,21,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Himad Abdelli,dz ALG,MF,Marseille,fr Ligue 1,26.0,1999.0,8,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Himad Abdelli,dz ALG,MF,Angers,fr Ligue 1,26.0,1999.0,13,11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Cell 3 — Basic dataset inspection
# This helps verify the size of the dataset and the available columns.

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (2779, 53)

Columns:
['Rk', 'Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 'Born', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'CrdY', 'CrdR', 'G+A-PK', 'Sh', 'SoT', 'SoT%', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT', 'PK_stats_shooting', 'PKatt_stats_shooting', 'Crs', 'TklW', 'Int', 'Fld', 'CrdY_stats_misc', 'CrdR_stats_misc', '2CrdY', 'Fls', 'OG', 'GA', 'GA90', 'SoTA', 'Saves', 'Save%', 'W', 'D', 'L', 'CS', 'CS%', 'PKatt_stats_keeper', 'PKA', 'PKsv', 'PKm']


In [5]:
# Cell 4 — Inspect player position distribution
# This is important because the scouting engine should compare players by role.

df["Pos"].value_counts()

Pos
MF       915
DF       697
FW       387
MF,FW    207
GK       178
FW,MF    147
DF,MF    133
MF,DF    111
DF,FW      3
Name: count, dtype: int64

## Champions League benchmark definition

The project needs a benchmark group: clubs that participated in the 2025/2026 UEFA Champions League.

Players from these clubs are treated as the **UCL-level reference group**.  
Players from other clubs are treated as potential **non-UCL scouting targets**.

In [6]:
# Cell 5 — Define Champions League clubs available in the dataset
# Club names must match exactly the names used in the dataset.

ucl_teams = [
    "Arsenal",
    "Athletic Club",
    "Atlético Madrid",
    "Atalanta",
    "Barcelona",
    "Bayern Munich",
    "Chelsea",
    "Dortmund",
    "Eintracht Frankfurt",
    "Inter",
    "Juventus",
    "Leverkusen",
    "Liverpool",
    "Manchester City",
    "Marseille",
    "Monaco",
    "Napoli",
    "Newcastle United",
    "Paris Saint-Germain",
    "Real Madrid",
    "Tottenham Hotspur",
    "Villarreal"
]

# Create a boolean flag identifying whether each player belongs to a UCL club.
df["is_ucl_team"] = df["Squad"].isin(ucl_teams)

df[["Player", "Squad", "is_ucl_team"]].head()

,Player,Squad,is_ucl_team
0,Brenden Aaronson,Leeds United,False
1,Zach Abbott,Nottingham Forest,False
2,Jones El-Abdellaoui,Celta Vigo,False
3,Himad Abdelli,Marseille,True
4,Himad Abdelli,Angers,False


In [7]:
# Cell 6 — Validate UCL flag on selected clubs
# This sanity check helps avoid wrong club classification.

df[df["Squad"].isin(["Atlético Madrid", "Milan", "Atalanta", "Athletic Club"])][
    ["Player", "Squad", "is_ucl_team"]
].drop_duplicates("Squad")

,Player,Squad,is_ucl_team
49,Honest Ahanor,Atalanta,True
84,Thiago Almada,Atlético Madrid,True
93,Yeray Álvarez,Athletic Club,True
165,Zachary Athekame,Milan,False


## Reliability filtering

Raw football data can be misleading when players have very few minutes.

To reduce noise, the first version of the model applies minimum sample thresholds:
- at least 15 appearances
- at least 900 minutes played

These thresholds can be adjusted in future versions.

In [8]:
# Cell 7 — Apply minimum sample thresholds
# This improves the reliability of per90 statistics.

MIN_APPEARANCES = 15
MIN_MINUTES = 900

df_filtered = df[
    (df["MP"] >= MIN_APPEARANCES) &
    (df["Min"] >= MIN_MINUTES)
].copy()

print("Original dataset:", df.shape)
print("Filtered dataset:", df_filtered.shape)

Original dataset: (2779, 54)
Filtered dataset: (1438, 54)


## Striker subset

The first version of the engine focuses on strikers (`FW`).

Future versions can extend this logic to:
- wingers
- midfielders
- defensive midfielders
- fullbacks
- centre-backs
- goalkeepers

In [9]:
# Cell 8 — Create striker subset
# We start with pure forwards only: Pos == "FW".

strikers = df_filtered[df_filtered["Pos"] == "FW"].copy()

print("Number of eligible strikers:", strikers.shape[0])
strikers.head()

Number of eligible strikers: 134


,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,W,D,L,CS,CS%,PKatt_stats_keeper,PKA,PKsv,PKm,is_ucl_team
19,20,Ragnar Ache,de GER,FW,Köln,de Bundesliga,27.0,1998.0,29,18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
21,22,Akor Adams,ng NGA,FW,Sevilla,es La Liga,26.0,2000.0,28,19,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
22,23,Che Adams,sct SCO,FW,Torino,it Serie A,29.0,1996.0,31,19,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
58,59,Ludovic Ajorque,fr FRA,FW,Brest,fr Ligue 1,32.0,1994.0,29,29,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
75,76,Alexandre Alemão,br BRA,FW,Rayo Vallecano,es La Liga,28.0,1998.0,22,10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False


In [10]:
# Cell 9 — Select base striker statistics
# These are the raw attacking features used to build derived per90 metrics.

striker_features = [
    "Gls",
    "Ast",
    "G+A",
    "Sh",
    "SoT",
    "G/Sh"
]

striker_data = strikers[
    [
        "Player",
        "Squad",
        "Comp",
        "MP",
        "Min",
        "90s",
        "is_ucl_team"
    ] + striker_features
].copy()

striker_data.head()

,Player,Squad,Comp,MP,Min,90s,is_ucl_team,Gls,Ast,G+A,Sh,SoT,G/Sh
19,Ragnar Ache,Köln,de Bundesliga,29,1718,19.1,False,7,4,11,52,22,0.13
21,Akor Adams,Sevilla,es La Liga,28,1846,20.5,False,8,3,11,55,27,0.09
22,Che Adams,Torino,it Serie A,31,1826,20.3,False,5,2,7,48,14,0.10
58,Ludovic Ajorque,Brest,fr Ligue 1,29,2550,28.3,False,7,9,16,47,15,0.15
75,Alexandre Alemão,Rayo Vallecano,es La Liga,22,969,10.8,False,2,0,2,19,9,0.11


## Feature engineering

Raw totals can be biased by playing time.

For scouting similarity, per90 metrics are more useful because they compare player output at a similar time scale.

Current engineered striker features:
- Goals per90
- Assists per90
- Shots per90
- Shots on target per90
- Goals per shot

In [11]:
# Cell 10 — Create per90 striker features
# These features compare players independently from total minutes played.

striker_data["Gls_per90"] = striker_data["Gls"] / striker_data["90s"]
striker_data["Ast_per90"] = striker_data["Ast"] / striker_data["90s"]
striker_data["Sh_per90"] = striker_data["Sh"] / striker_data["90s"]
striker_data["SoT_per90"] = striker_data["SoT"] / striker_data["90s"]

advanced_striker_features = [
    "Gls_per90",
    "Ast_per90",
    "Sh_per90",
    "SoT_per90",
    "G/Sh"
]

striker_data = striker_data.reset_index(drop=True)

striker_data.head()

,Player,Squad,Comp,MP,Min,90s,is_ucl_team,Gls,Ast,G+A,Sh,SoT,G/Sh,Gls_per90,Ast_per90,Sh_per90,SoT_per90
0,Ragnar Ache,Köln,de Bundesliga,29,1718,19.1,False,7,4,11,52,22,0.13,0.366492,0.209424,2.722513,1.151832
1,Akor Adams,Sevilla,es La Liga,28,1846,20.5,False,8,3,11,55,27,0.09,0.390244,0.146341,2.682927,1.317073
2,Che Adams,Torino,it Serie A,31,1826,20.3,False,5,2,7,48,14,0.10,0.246305,0.098522,2.364532,0.689655
3,Ludovic Ajorque,Brest,fr Ligue 1,29,2550,28.3,False,7,9,16,47,15,0.15,0.247350,0.318021,1.660777,0.530035
4,Alexandre Alemão,Rayo Vallecano,es La Liga,22,969,10.8,False,2,0,2,19,9,0.11,0.185185,0.000000,1.759259,0.833333


## Similarity model

The model uses:
- `StandardScaler` to normalize features
- `NearestNeighbors` to find statistically similar players
- Euclidean distance as the similarity distance metric

Lower distance means more similar.  
A derived `similarity_score` is added for easier interpretation.

In [12]:
# Cell 11 — Scale features and train Nearest Neighbors model
# Scaling prevents high-volume statistics from dominating the distance calculation.

X = striker_data[advanced_striker_features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = NearestNeighbors(
    n_neighbors=10,
    metric="euclidean"
)

model.fit(X_scaled)

,n_neighbors,10
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'euclidean'
,p,2
,metric_params,None
,n_jobs,None


In [13]:
# Cell 12 — Define reusable scouting function
# Given a UCL benchmark striker, the function returns similar non-UCL strikers.

def find_similar_non_ucl_players(player_name, n_neighbors=15):
    """
    Find non-UCL strikers with statistical profiles similar to a selected striker.

    Parameters
    ----------
    player_name : str
        Exact player name as written in the dataset.
    n_neighbors : int
        Number of nearest neighbors to retrieve before filtering non-UCL players.

    Returns
    -------
    pandas.DataFrame
        Non-UCL players ranked by similarity score.
    """

    player_match = striker_data[striker_data["Player"] == player_name]

    if player_match.empty:
        return f"Player '{player_name}' not found. Check spelling, accents, or position filter."

    player_index = player_match.index[0]

    distances, indices = model.kneighbors(
        [X_scaled[player_index]],
        n_neighbors=n_neighbors
    )

    similar_players = striker_data.iloc[indices[0]].copy()
    similar_players["distance"] = distances[0]

    # Convert distance into a more readable score.
    # Higher score = more similar profile.
    similar_players["similarity_score"] = 100 / (1 + similar_players["distance"])

    hidden_targets = similar_players[
        similar_players["is_ucl_team"] == False
    ].copy()

    return hidden_targets.sort_values("similarity_score", ascending=False)

## Example scouting report

Benchmark player: **Julián Álvarez**  
Club: **Atlético Madrid**  
Status: **Champions League club**

The engine searches for non-UCL strikers with similar statistical profiles.

In [14]:
# Cell 13 — Generate example scouting report

target_player = "Julián Álvarez"

results = find_similar_non_ucl_players(
    target_player,
    n_neighbors=15
)

results

,Player,Squad,Comp,MP,Min,90s,is_ucl_team,Gls,Ast,G+A,Sh,SoT,G/Sh,Gls_per90,Ast_per90,Sh_per90,SoT_per90,distance,similarity_score
0,Ragnar Ache,Köln,de Bundesliga,29,1718,19.1,False,7,4,11,52,22,0.13,0.366492,0.209424,2.722513,1.151832,0.503260,66.522074
1,Akor Adams,Sevilla,es La Liga,28,1846,20.5,False,8,3,11,55,27,0.09,0.390244,0.146341,2.682927,1.317073,0.635584,61.140245
107,Rômulo,RB Leipzig,de Bundesliga,28,2080,23.1,False,9,4,13,61,27,0.15,0.389610,0.173160,2.640693,1.168831,0.732426,57.722507
73,Rafael Leão,Milan,it Serie A,27,1771,19.7,False,9,3,12,60,23,0.12,0.456853,0.152284,3.045685,1.167513,0.870345,53.466080
117,Sambou Soumano,Lorient,fr Ligue 1,30,931,10.3,False,4,2,6,28,10,0.14,0.388350,0.194175,2.718447,0.970874,0.884124,53.075071
43,Karl Etta Eyong,Levante,es La Liga,28,1410,15.7,False,6,2,8,41,21,0.15,0.382166,0.127389,2.611465,1.337580,0.989833,50.255468
40,Emersonn,Toulouse,fr Ligue 1,26,1469,16.3,False,6,2,8,51,19,0.12,0.368098,0.122699,3.128834,1.165644,1.048974,48.804904
29,Keinan Davis,Udinese,it Serie A,27,1890,21.0,False,10,3,13,45,22,0.13,0.476190,0.142857,2.142857,1.047619,1.059026,48.566644
96,Mikel Oyarzabal,Real Sociedad,es La Liga,30,2482,27.6,False,14,3,17,74,34,0.11,0.507246,0.108696,2.681159,1.231884,1.082570,48.017600
103,Andrea Pinamonti,Sassuolo,it Serie A,34,2434,27.0,False,8,3,11,70,27,0.11,0.296296,0.111111,2.592593,1.000000,1.113224,47.321055


In [15]:
# Cell 14 — Export scouting report to processed data folder
# This creates a reusable CSV output for the repository.

os.makedirs("../data/processed", exist_ok=True)

output_path = "../data/processed/julian_alvarez_similar_non_ucl_strikers.csv"

results.to_csv(output_path, index=False)

print(f"Scouting report saved to: {output_path}")

Scouting report saved to: ../data/processed/julian_alvarez_similar_non_ucl_strikers.csv


In [16]:
# Cell 15 — Compact report view
# A cleaner output for quick interpretation.

results[
    [
        "Player",
        "Squad",
        "Comp",
        "MP",
        "Min",
        "90s",
        "Gls",
        "Ast",
        "G+A",
        "Gls_per90",
        "Ast_per90",
        "Sh_per90",
        "SoT_per90",
        "similarity_score"
    ]
].round(3)

,Player,Squad,Comp,MP,Min,90s,Gls,Ast,G+A,Gls_per90,Ast_per90,Sh_per90,SoT_per90,similarity_score
0,Ragnar Ache,Köln,de Bundesliga,29,1718,19.1,7,4,11,0.366,0.209,2.723,1.152,66.522
1,Akor Adams,Sevilla,es La Liga,28,1846,20.5,8,3,11,0.390,0.146,2.683,1.317,61.140
107,Rômulo,RB Leipzig,de Bundesliga,28,2080,23.1,9,4,13,0.390,0.173,2.641,1.169,57.723
73,Rafael Leão,Milan,it Serie A,27,1771,19.7,9,3,12,0.457,0.152,3.046,1.168,53.466
117,Sambou Soumano,Lorient,fr Ligue 1,30,931,10.3,4,2,6,0.388,0.194,2.718,0.971,53.075
43,Karl Etta Eyong,Levante,es La Liga,28,1410,15.7,6,2,8,0.382,0.127,2.611,1.338,50.255
40,Emersonn,Toulouse,fr Ligue 1,26,1469,16.3,6,2,8,0.368,0.123,3.129,1.166,48.805
29,Keinan Davis,Udinese,it Serie A,27,1890,21.0,10,3,13,0.476,0.143,2.143,1.048,48.567
96,Mikel Oyarzabal,Real Sociedad,es La Liga,30,2482,27.6,14,3,17,0.507,0.109,2.681,1.232,48.018
103,Andrea Pinamonti,Sassuolo,it Serie A,34,2434,27.0,8,3,11,0.296,0.111,2.593,1.000,47.321


## Next steps

Possible improvements:
1. Extend the engine to additional roles.
2. Add advanced FBref metrics such as xG, xA, progressive passes, and progressive carries.
3. Add market value and age to identify cost-efficient transfer targets.
4. Create radar charts for player comparison.
5. Build a Streamlit dashboard to make the scouting engine interactive.

In [18]:
import pandas as pd

full_df = pd.read_csv(
    "../data/raw/players_data_light-2025_2026.csv"
)

print(full_df.shape)

(2779, 53)
